# 05 — Joins and Saving Data

**Environment:** Ubuntu 22.04 · Python 3.12.3 · OpenJDK 17 · Spark 4.1.x · local PySpark.

Run cells in order. Each notebook can start a Spark session independently.

**Learning goal:** Use `inner`, `left`, `left_anti` joins, understand unmatched references, and save summarized results to Parquet and CSV.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'transactions.csv').is_file()), None)
assert ROOT is not None, 'Open the workshop folder in Jupyter before running this notebook.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from training_lib import create_spark, read_raw, typed_transactions, quality_condition, clean_transactions, completed_enriched, revenue_by_category
from pyspark.sql import functions as F
spark = create_spark('TG117-Workshop')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| master:', spark.sparkContext.master)

In [ ]:
raw, customers, products = read_raw(spark, ROOT)
clean = clean_transactions(raw)
completed = clean.filter(F.col('status') == 'COMPLETED')
print('Completed after basic validation:',completed.count())

### Identify orphan reference keys using left-anti joins

In [ ]:
unknown_products = completed.join(products.select('product_id'), 'product_id', 'left_anti')
unknown_customers = completed.join(customers.select('customer_id'), 'customer_id', 'left_anti')
unknown_products.select('transaction_id','product_id').show()
unknown_customers.select('transaction_id','customer_id').show()
assert unknown_products.count() == 1
assert unknown_customers.count() == 1

### Compare left and inner join

A left join preserves all rows from the left side; unmatched dimension fields become NULL. An inner join discards unmatched transactions.

In [ ]:
left_joined = completed.join(products, 'product_id', 'left')
print('Completed rows:',completed.count(),'left join rows:',left_joined.count())
left_joined.filter(F.col('category').isNull()).select('transaction_id','product_id','category').show()

### Enrich and save category summary

In [ ]:
enriched = completed_enriched(clean, customers, products)
category_report = revenue_by_category(enriched)
category_report.show(truncate=False)
parquet_dir = ROOT / 'output' / 'category_revenue_parquet'
csv_dir = ROOT / 'output' / 'category_revenue_csv'
category_report.write.mode('overwrite').parquet(str(parquet_dir))
category_report.write.mode('overwrite').option('header',True).csv(str(csv_dir))
assert spark.read.parquet(str(parquet_dir)).count() == 4
assert spark.read.option('header',True).csv(str(csv_dir)).count() == 4
print('Saved under:', ROOT / 'output')

**Operational note:** Spark creates directories with multiple `part-*` files and metadata. Never overwrite `data/` or an existing production path; the workshop writes only beneath `output/`.